In [ ]:
!pip install transformers
!pip install pyannote.audio
!pip install ipywidgets
!pip install pydub

In [5]:
import torch
import torchaudio
#from .autonotebook import tqdm as notebook_tqdm
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from pyannote.audio import Pipeline
#from whisperx_numpy2_compatibility import load_align_model, align
#from whisperx_numpy2_compatibility.diarize import assign_word_speakers
#from pyannote.core import Segment


import textwrap
import os
import logging

#os.environ['CURL_CA_BUNDLE'] = ''

In [6]:
def find_intersections(speakers, texts):
    intersections = []

    for text in texts:
        text_start, text_end = text['start'], text['end']-0.1
        for turn, _, speaker in speakers.itertracks(yield_label=True):
            speaker_start, speaker_end = turn.start, turn.end
            
            # Find the overlap between the speaker's interval and the text's interval
            start = max(text_start, speaker_start)
            end = min(text_end, speaker_end)
            
            if start < end:  # There is an intersection
                if intersections and intersections[-1]['speaker'] == speaker:
                    intersections[-1]['end'] = end
                    intersections[-1]['text'] += ' ' + text['text']
                else:
                    intersections.append({
                        'start': start,
                        'end': end,
                        'speaker': speaker,
                        'text': text['text']
                    })
    return intersections


In [7]:
LOCAL_MODEL = False

In [8]:
def merge_speech_segments(segments):
    merged_segments = []
    for segment in segments:
        if merged_segments and segment["speaker"] == merged_segments[-1]["speaker"]:
            # Extend the end time and append text for the same speaker
            merged_segments[-1]["end"] = segment["end"]
            merged_segments[-1]["text"] += " " + segment["text"]
        else:
            # Add a new segment if the speaker changes
            merged_segments.append(segment)
    return merged_segments


In [9]:
def save_speech_to_file_with_indent(segments, filename):
    with open(filename, "w", encoding="utf-8") as file:
        for segment in segments:
            # Format the speaker tag
            speaker_tag = f"{segment['speaker'].upper()}:\n"
            
            # Wrap the text to 128 characters and indent each line
            wrapped_text = textwrap.fill(segment["text"], width=128, subsequent_indent="    ")
            
            # Write the formatted text to the file
            file.write(speaker_tag)
            file.write(wrapped_text)
            file.write("\n\n")  # Add a blank line between speakers


In [10]:
HF_TOKEN="XXXXXX"

if LOCAL_MODEL:
    DIARIZATION_MODEL="/Projects/AI/models/speaker-diarization-3.1/config.yaml"
    align_model="/Projects/AI/models/wav2vec2-large-xlsr-53-russian/"
else:
    DIARIZATION_MODEL="pyannote/speaker-diarization-3.1"
    align_model='jonatasgrosman/wav2vec2-large-xlsr-53-russian'

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
print(device)

cpu


In [12]:
#Initializing up wisper pipeline
whisper_model_id="openai/whisper-large-v3"
#whisper_model_id="openai/whisper-medium"
whisper_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    whisper_model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
whisper_model.config.forced_decoder_ids = None
whisper_model.to(device)
whisper_processor = AutoProcessor.from_pretrained(
    whisper_model_id
)
whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model=whisper_model,
    tokenizer=whisper_processor.tokenizer,
    feature_extractor=whisper_processor.feature_extractor,
    chunk_length_s=30,  # Process audio in 30-second chunks
    stride_length_s=10,  # Optional overlap between chunks    
    torch_dtype=torch_dtype,
    device=device,
)


Device set to use cpu


In [13]:
diarization_pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1")
if torch.cuda.is_available():
    diarization_pipeline.to(torch.device("cuda"))
#model = whisper.load_model(WHISPER_MODEL, download_root='./models', device=DEVICE)

INFO:speechbrain.utils.quirks:Applied quirks (see `speechbrain.utils.quirks`): [allow_tf32, disable_jit_profiling]
INFO:speechbrain.utils.quirks:Excluded quirks specified by the `SB_DISABLE_QUIRKS` environment (comma-separated list): []
C:\Program Files\Python312\Lib\inspect.py:1007: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  if ismodule(module) and hasattr(module, '__file__'):


In [14]:
def transcript(file_name):
    logging.info('started')
    #audio, sample_rate = torchaudio.load(file_name, backend='soundfile')

    script = whisper_pipe(file_name, return_timestamps='word', generate_kwargs={"language": "russian"})
    with open('script_2.txt', "w", encoding="utf-8") as f:
        f.write(script["text"])    
    logging.info('loaded')
    diarized = diarization_pipeline(file_name, min_speakers=1, max_speakers=9)
    logging.info(diarized)
    # Combine results
    speaker_transcription = []
    for chunk in script['chunks']:
        start_time, end_time = chunk["timestamp"][0], chunk["timestamp"][1]
        speaker = "Unknown"
        for turn, _, speaker_label in diarized.itertracks(yield_label=True):
            if turn.start <= start_time <= turn.end or turn.start <= end_time <= turn.end :
                speaker = speaker_label
                break
        speaker_transcription.append({
            "start": start_time,
            "end": end_time,
            "speaker": speaker,
            "text": chunk["text"]
        })
    transcribed = []
    for segment in speaker_transcription:
        transcribed.append(
            {
                "start": segment["start"],
                "end": segment["end"],
                "text": segment["text"],
                "speaker": segment["speaker"] if 'speaker' in segment else "ND"
            }
        )

    merged = merge_speech_segments(transcribed)

    out_file, _ = os.path.splitext(file_name)
    out_file = f"{out_file}_transcript_2.txt"
    save_speech_to_file_with_indent(merged, out_file)

In [15]:
audios=["audio/audio1097921934.wav"]#, "./audio/audio1415011527.m4a", "./audio/audio1499365096.m4a"]

In [16]:
for audio in audios:
        transcript(audio)

INFO:root:started


TypeError: We expect a numpy ndarray as input, got `<class 'torch.Tensor'>`

In [2]:
torchaudio.list_audio_backends()

['soundfile']